# Week 9 - Forms

We build activation functions, explain why a hidden layer needs a nonlinearity, and apply the Forms to MNIST. The original Week 9 source is preserved in the cells below. Separate notebook cells run the experiments without creating images.

## Setup

Run from this repository with Python 3, NumPy, and Matplotlib. Keep this notebook beside the Week 9 Python files. The loader can download MNIST if it is absent; setup uses the existing repository copy when available. Only the three original required images are linked below. No cell creates a figure or saves an image.

In [40]:
from pathlib import Path
import os, sys

_candidates = [Path.cwd(), Path.cwd() / "week9"]
week9_dir = next(p for p in _candidates if (p / "part3_upgrade.py").is_file())
repo_dir = week9_dir.parent
sys.path.insert(0, str(week9_dir.resolve()))
# The source loader uses a relative mnist.npz path.
os.chdir(repo_dir if (repo_dir / "mnist.npz").exists() else week9_dir)
# Preserve source guards but skip image writes and repeated test scoring.
__name__ = "week9_notebook"


## Data loader

The loader already divides pixels by 255. Dividing again would shrink the inputs unnecessarily. Validation uses rows 1000-1999 before training is sliced to rows 0-999.

### Original source: `data.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [41]:
import os, urllib.request, numpy as np

URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"

def load_mnist(path="mnist.npz"):
    if not os.path.exists(path):
        urllib.request.urlretrieve(URL, path)
    d = np.load(path)
    X_train = d["x_train"].reshape(len(d["x_train"]), 784) / 255.0
    X_test = d["x_test"].reshape(len(d["x_test"]), 784) / 255.0
    return (X_train, d["y_train"]), (X_test, d["y_test"])

def one_hot(labels, n=10):
    out = np.zeros((len(labels), n))
    out[np.arange(len(labels)), labels] = 1
    return out

## Part 1 - Build the Forms

Sigmoid maps into (0, 1), tanh into (-1, 1), and ReLU into [0, infinity). Their derivatives reuse the forward output: sigmoid uses `out * (1 - out)`, tanh uses `1 - out**2`, and ReLU uses whether the output is positive. This avoids recalculating the forward function. ReLU uses slope zero at zero.

Softmax subtracts each row maximum before exponentiating. This leaves the probabilities unchanged while avoiding overflow. Each row sums to one.

### Original source: `forms.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [42]:
import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def sigmoid_deriv(out):
    return out * (1 - out)  # takes the OUTPUT


def tanh(x):
    return np.tanh(x)


def tanh_deriv(out):
    return 1 - out**2  # takes the OUTPUT


def relu(x):
    return (x > 0) * x


def relu_deriv(out):
    return out > 0  # takes the OUTPUT


def softmax(z):  # z.shape == (batch, k)
    z = z - np.max(z, axis=1, keepdims=True)  # subtract row max
    exp_z = np.exp(z)  # exp(onentiate)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)  # divide by row sum


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    x = np.linspace(-6, 6, 400)
    plt.plot(x, sigmoid(x), color="gray", label="sigmoid (0, 1)")
    plt.plot(x, tanh(x), color="blue", label="tanh (-1, 1)")
    plt.plot(x, relu(x), color="orange", label="relu [0, inf)")
    plt.axhline(0, color="black", lw=0.5)
    plt.xlabel("input x")
    plt.ylabel("Form output")
    plt.legend()
    plt.title("Three Forms")
    plt.savefig("forms_plot.png")
    plt.show()


![Three activation Forms](forms_plot.png)

## Part 2 - Prove You Need a Form

Associativity gives `(X @ W1) @ W2 = X @ (W1 @ W2)`. ReLU introduces an input-dependent bend at zero. Signed inputs and weights ensure that this bend can affect the demonstration. The least-squares residual checks whether even the best linear fit can reproduce all five examples.

### Original source: `part2_nonlinearity.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [43]:
import numpy as np
from forms import relu

def main():
    np.random.seed(1)
    X = np.random.randn(5, 4)
    W1 = np.random.randn(4, 3)
    W2 = np.random.randn(3, 2)

    Y_two = (X @ W1) @ W2

    W_prime = W1 @ W2
    Y_one = X @ W_prime

    print("Do the two linear layers match the collapsed layer?")
    print(np.allclose(Y_two, Y_one))
    print("Maximum absolute difference:", np.max(np.abs(Y_two - Y_one)))

    Y_nl = relu(X @ W1) @ W2

    print("\nDoes inserting ReLU still match the collapsed linear layer?")
    print(np.allclose(Y_nl, Y_one))
    print("Maximum absolute difference:", np.max(np.abs(Y_nl - Y_one)))

    
    W_best, residuals, rank, singular_values = np.linalg.lstsq(
        X, Y_nl, rcond=None
    )
    best_linear_output = X @ W_best
    residual_norm = np.linalg.norm(Y_nl - best_linear_output)

    print("\nResidual from the best-fit single linear transformation:")
    print(residual_norm)
    print("Is the residual nonzero?", residual_norm > 1e-9)

if __name__ == "__main__":
    main()

# Without ReLU, the two weight matrices can be combined into one because
# matrix multiplication is linear. ReLU changes this by setting negative
# values to zero while keeping positive values, so the second layer receives
# different values than it would from a purely linear transformation.
# Therefore, the two layers can no longer be collapsed into one equivalent
# linear transformation for all possible inputs.

In [44]:
main()

Do the two linear layers match the collapsed layer?
True
Maximum absolute difference: 1.3322676295501878e-15

Does inserting ReLU still match the collapsed linear layer?
False
Maximum absolute difference: 3.3433278701904103

Residual from the best-fit single linear transformation:
0.08642042721114628
Is the residual nonzero? True


## Part 3 - The Right Form per Layer

The network is 784 -> 100 -> 10, with seed 1, alpha 0.02, 300 epochs, and batches of 100. The slope is saved before dropout. Both deltas use the old weights; backward dropout uses the exact `mask * 2` convention. Softmax produces class probabilities, while raw-score argmax gives the same predicted class during evaluation.

**Preserved-code limitation:** the original training loop broadcasts one `(100,)` mask across a batch, rather than drawing the required `(batch, 100)` mask. It also discards incomplete batches. All 1,000 training rows are used here because 100 divides 1,000. These are properties of the original implementation, not changes made during notebook assembly.

### Original source: `part3_upgrade.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [45]:
import numpy as np
from data import load_mnist, one_hot
import matplotlib.pyplot as plt
from forms import tanh, tanh_deriv, softmax

# Load MNIST dataset as 3 sets: 1000-long training set, 1000-long validation set,
# and unknown-length test set.
(X_train, y_train), (X_test, y_test) = load_mnist()
X_val, y_val = X_train[1000:2000], one_hot(y_train[1000:2000])
X_train, y_train = X_train[:1000], one_hot(y_train[:1000])
X_test, y_test = X_test, one_hot(y_test)

# Record default seed and various variables
np.random.seed(1) # seed FIRST, then initialise
alpha, epochs, hidden_size, batch_size = (0.02, 300, 100, 100)
# Roll up our random weights: one a 784 x 100 matrix intaking
# all values from an image, as random values from -.01 to .01,
# one a 100 x 10 matrix intaking the hidden layer and outputting
# log-probs for each of the possible numbers, as random values
# from -.1 to .1
W_0_1 = 0.02 * np.random.random((784, hidden_size)) - 0.01
W_1_2 = 0.2 * np.random.random((hidden_size, 10)) - 0.1

# Run a single training step over a full batch, updating
# weight matrices in place, taking in a single mask for the
# whole batch from the external function, using any provided
# activation functions on the hidden layer.
def train_step(layer_0, y_batch, W_0_1, W_1_2, mask,
alpha, hidden_f=tanh, hidden_d=tanh_deriv):
    # calculate layer 1
    layer_1 = hidden_f(layer_0 @ W_0_1)
    # save the derivative of layer 1 before moving on; we'll use
    # it later.
    slope = hidden_d(layer_1)

    # intentionally uses the same mask for all in the batch
    layer_1 *= mask * 2

    # calculate layer 2, convert from log probs to full probs
    layer_2 = softmax(layer_1 @ W_1_2)

    # count how many are in the batch so we can divide by that much
    batch_size = layer_0.shape[0]

    # calculate for the whole batch
    delta_2 = (layer_2 - y_batch) / batch_size

    # here's where we use slope from earlier
    delta_1 = delta_2 @ W_1_2.T * slope

    # scale in our mask
    delta_1 *= mask * 2

    # weight updates
    W_1_2 -= alpha * layer_1.T @ delta_2
    W_0_1 -= alpha * layer_0.T @ delta_1

    # a little odd that we return this, but it's in the assignment spec
    return delta_1


def evaluate(X, y, W_0_1, W_1_2, hidden_f=tanh):
    """Runs through a set of two weights W_0_1 and W_1_2, applying the
    listed activation function to layer 1, for each X in set X, checking
    final output against the corresponding y in set y. Does not use
    softmax on the output; this does not influence the result.
    Returns the fraction of input that was correctly predicted."""
    correct = 0
    for i in range(len(X)):
        layer_1 = hidden_f(X[i:i+1].dot(W_0_1))
        layer_2 = layer_1.dot(W_1_2)
        correct += int(np.argmax(layer_2) == np.argmax(y[i]))
    
    return correct / len(X)




def train_forms(X_train, y_train, X_val, y_val, alpha = 0.005, epochs=350, batch_size=100):

    val_hist = []

    # For each epoch...
    for _ in range(epochs):

        # note that this discards incomplete batches, e.g. 
        # batch size 100, total size 1079 will ditch the
        # last 79 entries

        # for each mini-batch...
        for i in range(len(X_train) // batch_size):
            # figure out the bounds of that mini-batch...
            start, end = i * batch_size, (i+1) * batch_size

            # build a mask for that mini-batch...
            mask = np.random.randint(2, size=hidden_size) # ~50% ones, ~50% zeros

            # and chuck it into our helper function that does everything for us.
            train_step(X_train[start:end], y_train[start:end], W_0_1, W_1_2, mask, alpha, tanh, tanh_deriv)



        # After that epoch, calculate accuracy on the new weights and record it
        val_hist.append(evaluate(X_val, y_val, W_0_1, W_1_2))

        # print(f"Epoch {it+1}/{epoch} - Train Acc: {evaluate(X_train, y_train, W_0_1, W_1_2):.4f}, Val Acc: {evaluate(X_val, y_val, W_0_1, W_1_2):.4f}")
    return val_hist

if __name__ == "__main__":
    # run the training!
    val_hist_batch = train_forms(X_train, y_train, X_val, y_val, alpha, epochs, batch_size)

    # check it on our test set!
    final_test_score = evaluate(X_test, y_test, W_0_1, W_1_2)

    # And print that out!
    print(f"Final score after {epochs} epochs on the test set: {final_test_score}")


    # ---- Plot ----
    plt.plot(range(1, epochs + 1), val_hist_batch, label="Validation Accuracy with Batching")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Validation Accuracy By Epoch")
    plt.legend()
    plt.grid(True)

    plt.savefig("week9/upgrade_plot.png")
    plt.show()

In [46]:
# Run the original Part 3 routine, without its plotting block.
val_hist_batch = train_forms(X_train, y_train, X_val, y_val, alpha, epochs, batch_size)
part3_validation = val_hist_batch[-1]
# The notebook's only full test-set evaluation.
final_test_score = evaluate(X_test, y_test, W_0_1, W_1_2)
print(f"Part 3 final validation accuracy: {part3_validation:.2%}")
print(f"Part 3 final test accuracy: {final_test_score:.2%}")
print(f"Change from Week 8 final batched validation: {(part3_validation - 0.842) * 100:+.2f} percentage points")


Part 3 final validation accuracy: 87.80%
Part 3 final test accuracy: 86.61%
Change from Week 8 final batched validation: +3.60 percentage points


![Original Part 3 validation curve](upgrade_plot.png)

Compare final validation with final validation, rather than a peak or a test score. The existing recorded Part 3 result is 87.80% validation and 86.61% test, versus Week 8's 84.2% validation: a 3.6 percentage-point increase. Width also increased from 40 to 100 hidden units, so this comparison cannot attribute the gain to tanh alone. Executed output above is the result of the current rerun; linked images remain the original files.

## Part 4 - Compare the Forms

We reuse `train_step` with each activation and its derivative. The following source is unchanged. Its original main block also evaluates the test set and saves a plot; the notebook deliberately skips that block and runs training plus validation only.

**Comparison limitation:** the original source uses different initialization distributions for tanh, ReLU, and sigmoid and advances one random stream across runs. Width, data, epochs, alpha, and batching are shared, but initialization and dropout draws are not controlled identically. These numbers describe the existing runs; they do not isolate activation as the only cause.

### Original source: `part4_compare.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [47]:
import numpy as np
from data import load_mnist, one_hot
import matplotlib.pyplot as plt
from part3_upgrade import train_step, evaluate

from forms import sigmoid, sigmoid_deriv, tanh, tanh_deriv, relu, relu_deriv

(X_train, y_train), (X_test, y_test) = load_mnist()
X_val, y_val = X_train[1000:2000], one_hot(y_train[1000:2000])
X_train, y_train = X_train[:1000], one_hot(y_train[:1000])
X_test, y_test = X_test, one_hot(y_test)

np.random.seed(1) 
alpha, epochs, hidden_size, batch_size = (0.02, 300, 100, 100)

W_0_1_TANH = 0.02 * np.random.random((784, hidden_size)) - 0.01
W_1_2_TANH = 0.2 * np.random.random((hidden_size, 10)) - 0.1

W_0_1_RELU = np.random.random((784, hidden_size)) * np.sqrt(2 / 784)
W_1_2_RELU = np.random.random((hidden_size, 10)) * np.sqrt(2 / hidden_size)

# Glorot for sigmoid. I remembered we mentioned this briefly in ML.
W_0_1_SIG = np.random.randn(784, hidden_size) * np.sqrt(2 / (784 + hidden_size))
W_1_2_SIG = np.random.randn(hidden_size, 10) * np.sqrt(2 / (hidden_size + 10))



# This is taken from Part 3 and modified to be more modular for the types of activation functions.
def train_forms(X_train, y_train, X_val, y_val, w_0_1, w_1_2, alpha = 0.005, epochs=350, batch_size=100, func=tanh, func_deriv=tanh_deriv):
    val_hist = []
    # For each epoch...
    for _ in range(epochs):
        for i in range(len(X_train) // batch_size):
            start, end = i * batch_size, (i+1) * batch_size
            mask = np.random.randint(2, size=hidden_size)
            train_step(X_train[start:end], y_train[start:end], w_0_1, w_1_2, mask, alpha, func, func_deriv)

        val_hist.append(evaluate(X_val, y_val, w_0_1, w_1_2, func))

        # print(f"Epoch {it+1}/{epoch} - Train Acc: {evaluate(X_train, y_train, W_0_1, W_1_2):.4f}, Val Acc: {evaluate(X_val, y_val, W_0_1, W_1_2):.4f}")
    return val_hist



if __name__ == "__main__":
    tanh_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_TANH, W_1_2_TANH, alpha, epochs, batch_size, tanh, tanh_deriv)

    tanh_final_test_score = evaluate(X_test, y_test, W_0_1_TANH, W_1_2_TANH, tanh)

    print(f"Final score after {epochs} epochs on the tanh test set: {tanh_final_test_score}")


    relu_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_RELU, W_1_2_RELU, alpha, epochs, batch_size, relu, relu_deriv)
    
    relu_final_test_score = evaluate(X_test, y_test, W_0_1_RELU, W_1_2_RELU, relu)

    print(f"Final score after {epochs} epochs on the relu test set: {relu_final_test_score}")


    sig_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_SIG, W_1_2_SIG, alpha, epochs, batch_size, sigmoid, sigmoid_deriv)
    
    sig_final_test_score = evaluate(X_test, y_test, W_0_1_SIG, W_1_2_SIG, sigmoid)

    print(f"Final score after {epochs} epochs on the sig test set: {sig_final_test_score}")






    # ---- Plot ----
    plt.plot(range(1, epochs + 1), tanh_hist_batch, label="TANH Accuracy")
    plt.plot(range(1, epochs + 1), relu_hist_batch, label="RELU Accuracy")
    plt.plot(range(1, epochs + 1), sig_hist_batch, label="SIGMOID Accuracy")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Validation Accuracy By Activation Function")
    plt.legend()
    plt.grid(True)

    plt.savefig("compare_plot.png")
    plt.show()





In [48]:
# Run the three original training calls in their original order.
# No test-set evaluation and no image generation.
tanh_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_TANH, W_1_2_TANH, alpha, epochs, batch_size, tanh, tanh_deriv)
relu_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_RELU, W_1_2_RELU, alpha, epochs, batch_size, relu, relu_deriv)
sig_hist_batch = train_forms(X_train, y_train, X_val, y_val, W_0_1_SIG, W_1_2_SIG, alpha, epochs, batch_size, sigmoid, sigmoid_deriv)
print("Hidden Form | Final validation accuracy")
print("------------|--------------------------")
for name, history in [("tanh", tanh_hist_batch), ("ReLU", relu_hist_batch), ("sigmoid", sig_hist_batch)]:
    print(f"{name:11} | {history[-1]:.2%}")


Hidden Form | Final validation accuracy
------------|--------------------------
tanh        | 87.60%
ReLU        | 88.00%
sigmoid     | 84.30%


| Hidden Form | Original recorded final validation accuracy |
|---|---:|
| tanh | 87.6% |
| ReLU | 88.0% |
| sigmoid | 84.3% |

![Original activation comparison](compare_plot.png)

The table is Markdown, not an image. Sigmoid trails tanh by 3.3 points in the recorded comparison. Its derivative peaks at 0.25, compared with 1 for tanh, so hidden updates can be smaller at the same learning rate. Sigmoid's positive outputs also change optimization. These mechanisms are relevant, but the differing initializations prevent a claim that activation alone caused the measured gap.

## Part 5 - Tests

The original test file is preserved below. It discovers five tests, rather than the assignment's minimum seven. Its accuracy comparison scores the test set, so we leave that function defined but do not call it here. The notebook runs the four original mathematical checks, adds finite-difference and one-step backpropagation checks, and uses a matched-seed validation-only comparison as the seventh check. These supplemental checks do not modify `test_forms.py`.

### Original source: `test_forms.py`

The following cell preserves the complete Week 9 file, including its original script-only block. Setup keeps `__name__` different from `"__main__"`, so assembly does not run plot-saving or extra test-set scoring blocks.

In [49]:
import numpy as np
from data import load_mnist, one_hot
import forms as av  # activation function
from part2_nonlinearity import main
from part3_upgrade import train_step, evaluate
from part4_compare import train_forms


def test_softmax_distro():
    # Part 1: checksums
    softmax_sum_test()
    softmax_sum_test(shape=(6, 7))
    softmax_sum_test(seed=42)
    softmax_sum_test(42, (6, 7))

    # Part 2: finity and affinity
    softmax_finite_test(np.array([[1000.0, 1001.0]]), np.array([[0.269, 0.731]]))


def test_ranges():
    x = np.linspace(-15, 15, 1001)
    sigmoid_x = av.sigmoid(x)
    tanh_x = av.tanh(x)
    relu_x = av.relu(x)

    assert (
        np.min(sigmoid_x) > 0 and np.max(sigmoid_x) < 1
    ), f"Sigmoid must be in bounds (0, 1) -- got min={np.min(sigmoid_x)}, max={np.max(sigmoid_x)}"
    assert (
        np.min(tanh_x) > -1 and np.max(tanh_x) < 1
    ), f"Tanh must be in bounds (-1, 1) -- got min={np.min(tanh_x)}, max={np.max(tanh_x)}"
    assert np.all(relu_x >= 0), f"ReLU must be >= 0, got {relu_x}"

    x_idx = np.where(x > 0, 1, 0)
    relu_x_idx = np.where(x != 0, 1, 0)
    assert [
        a == b for a, b in zip(x, relu_x) if a > 0
    ], f"ReLU must have matching indices -- expected {x_idx}, got {relu_x_idx}"


def test_relu_and_slope():
    x = np.linspace(-15, 15, 1001)
    xa = av.relu(x)
    neg_idx = np.asarray([i for i in range(len(xa)) if xa[i] < 0])
    assert np.all(
        np.array([xa[i] == 0 for i in neg_idx])
    ), f"ReLU should be 0 for every negative x -- got  {xa.tolist()}"
    assert np.array_equal(
        av.relu_deriv(xa), x > 0
    ), f"ReLU's slope should match the input's sign -- got {xa.tolist()}"


def test_lin_layers_collapse():
    np.random.seed(1)
    X = np.random.randn(5, 4)
    W1 = np.random.randn(4, 3)
    W2 = np.random.randn(3, 2)

    assert np.allclose(
        (X @ W1) @ W2, X @ (W1 @ W2), rtol=1e-9, atol=1e-9
    ), f"Input with weights should be associative"
    assert not np.allclose(
        av.relu(X @ W1) @ W2, X @ (W1 @ W2)
    ), f"ReLU should break linearity"


def test_compare_tanh_sigmoid():
    (X_train, y_train), (X_test, y_test) = load_mnist()
    X_val, y_val = X_train[1000:2000], one_hot(y_train[1000:2000])
    X_train, y_train = X_train[:1000], one_hot(y_train[:1000])
    X_test, y_test = X_test, one_hot(y_test)

    np.random.seed(1)
    alpha, epochs, hidden_size, batch_size = (0.02, 30, 100, 100)

    W_0_1_TANH = 0.02 * np.random.random((784, hidden_size)) - 0.01
    W_1_2_TANH = 0.2 * np.random.random((hidden_size, 10)) - 0.1

    # Glorot for sigmoid. I remembered we mentioned this briefly in ML.
    W_0_1_SIG = np.random.randn(784, hidden_size) * np.sqrt(2 / (784 + hidden_size))
    W_1_2_SIG = np.random.randn(hidden_size, 10) * np.sqrt(2 / (hidden_size + 10))

    train_forms(
        X_train,
        y_train,
        X_val,
        y_val,
        W_0_1_TANH,
        W_1_2_TANH,
        alpha,
        epochs,
        batch_size,
        av.tanh,
        av.tanh_deriv,
    )

    tanh_final_test_score = evaluate(X_test, y_test, W_0_1_TANH, W_1_2_TANH, av.tanh)

    train_forms(
        X_train,
        y_train,
        X_val,
        y_val,
        W_0_1_SIG,
        W_1_2_SIG,
        alpha,
        epochs,
        batch_size,
        av.sigmoid,
        av.sigmoid_deriv,
    )

    sig_final_test_score = evaluate(X_test, y_test, W_0_1_SIG, W_1_2_SIG, av.sigmoid)

    assert (
        tanh_final_test_score > sig_final_test_score
    ), f"Tanh final validation accuracy should be greater than sigmoid -- got tanh_final_val_acc={tanh_final_test_score}, sigmoid_final_val_acc={sig_final_test_score}"


def softmax_sum_test(seed=1, shape=(8, 10)):
    np.random.seed(seed)
    X = np.random.randn(shape[0], shape[1])
    Xa = av.softmax(X)
    Xsums = np.sum(Xa, axis=1)

    # Test: each row sums to 1; thus use axis=1 to sum the rows
    assert np.allclose(
        Xsums, 1, atol=1e-9, rtol=1e-9
    ), f"Softmax needs a stable distribution -- got {Xsums.tolist()}"


def softmax_finite_test(X, expected):
    Xa = av.softmax(X)
    # Test 1: no nan
    assert not np.any(np.isnan(X)), f"X cannot have NaN -- got {np.isnan(X)}"
    # Test 2: close to expected
    assert np.allclose(
        Xa, expected, rtol=0.02, atol=0.02
    ), f"Must be close to the expected value -- expected {expected}, got {Xa}"


# Copied from other tests
if __name__ == "__main__":
    tests = [name for name in dir() if name.startswith("test_")]
    for test_name in sorted(tests):
        test_func = globals()[test_name]
        try:
            test_func()
            print(f" PASS: {test_name}")
        except AssertionError as e:
            print(f" FAIL (AssertionError): {test_name} -- {e}")


In [50]:
def notebook_derivatives():
    x = np.array([-3., -1., 0., 1., 3.])
    h = 1e-5
    for f, derivative in [(tanh, tanh_deriv), (sigmoid, sigmoid_deriv)]:
        assert np.allclose(derivative(f(x)), (f(x+h)-f(x-h))/(2*h), atol=1e-4, rtol=0)

def notebook_backprop():
    rng = np.random.RandomState(0)
    x = rng.rand(4, 6)
    a, b = rng.randn(6, 5), rng.randn(5, 3)
    labels = np.eye(3)[[0, 2, 1, 2]]
    mask = rng.randint(2, size=(4, 5))
    hidden = np.tanh(x @ a)
    dropped = hidden * mask * 2
    d2 = (softmax(dropped @ b) - labels) / 4
    d1 = (d2 @ b.T) * (1 - hidden**2) * mask * 2
    aa, bb = a.copy(), b.copy()
    got = train_step(x, labels, aa, bb, mask, 0.02)
    assert np.allclose(got, d1)
    assert np.allclose(aa, a - 0.02 * x.T @ d1)
    assert np.allclose(bb, b - 0.02 * dropped.T @ d2)

def notebook_validation_comparison():
    scores = {}
    for name, f, d in [("tanh", tanh, tanh_deriv), ("sigmoid", sigmoid, sigmoid_deriv)]:
        np.random.seed(1)
        a = 0.02 * np.random.random((784, 100)) - 0.01
        b = 0.2 * np.random.random((100, 10)) - 0.1
        history = train_forms(X_train, y_train, X_val, y_val, a, b, 0.02, 30, 100, f, d)
        scores[name] = history[-1]
    print("30-epoch validation comparison:", scores)
    assert scores["tanh"] > scores["sigmoid"]

checks = [test_softmax_distro, test_ranges, test_relu_and_slope,
          test_lin_layers_collapse, notebook_derivatives, notebook_backprop,
          notebook_validation_comparison]
for check in checks:
    check()
    print("PASS:", check.__name__)
print(f"{len(checks)}/{len(checks)} notebook checks passed")


PASS: test_softmax_distro
PASS: test_ranges
PASS: test_relu_and_slope
PASS: test_lin_layers_collapse
PASS: notebook_derivatives
PASS: notebook_backprop
30-epoch validation comparison: {'tanh': 0.803, 'sigmoid': 0.248}
PASS: notebook_validation_comparison
7/7 notebook checks passed


## Submission notes

The original `.py` files are unchanged. Before claiming complete assignment compliance, the team still needs to address per-example dropout masks, remainder batches, matched Part 4 initialization and seeds, and the standalone test file's missing derivative/backpropagation checks and test-set comparison. The notebook supplements documentation and checks without silently rewriting the team's implementations.

Only `forms_plot.png`, `upgrade_plot.png`, and `compare_plot.png` are referenced. The existing `compare_curves.png` is unused; assembly neither creates nor deletes it.

## Holocron — Reading

Our Part 4 results were 87.6% validation accuracy for tanh, 88.0% for ReLU, and 84.3% for sigmoid. The network size and data stayed the same, yet the training outcomes changed. A Form changes both the hidden representation and the slopes that carry error backward. Sigmoid's derivative cannot exceed 0.25, while tanh's can reach 1. At the same learning rate, sigmoid can therefore make smaller hidden-layer updates. Its outputs are also always positive. That does not make sigmoid useless; it means this training setup may suit it less well.

Our comparison also used different initializations, so we cannot assign the entire accuracy difference to the Form alone. Part 3 reached 87.8% validation and 86.61% test accuracy, but its improvement over Week 8's 84.2% validation also came with a wider hidden layer. Those details matter when interpreting success.

Korrith's fragment describes the same danger as treating one activation as universally best. The Verach perfect a decisive Form until they stop recognizing situations that require another response. A network designer makes a similar mistake by using one activation everywhere without considering its range, slope, or purpose. Softmax suits competing output classes; hidden layers need useful representations and gradients. Mastery means understanding those constraints and choosing for the task, rather than confusing familiarity with a universal answer.